# ACE mouse tiny example

This notebook runs ACE on a small real mouse droplet single-cell dataset sampled from the Tabula Muris Senis / CZ CELLxGENE workflow used in the manuscript. It is intended as a reproducible example for the public repository, not as a full benchmark.

The notebook calls the command-line workflow in `examples/mouse_aging_workflow.py`, then displays the age-prediction and representation-quality metrics.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if not (repo_root / "examples" / "mouse_aging_workflow.py").exists():
    raise RuntimeError("Run this notebook from the ACE repository root or notebook/ directory.")

adata_rel = Path("data") / "ace_mouse_tiny.h5ad"
out_rel = Path("results") / "mouse_tiny_notebook"
adata_path = repo_root / adata_rel
out_dir = repo_root / out_rel
print("dataset:", adata_rel)
print("output:", out_rel)

dataset: data/ace_mouse_tiny.h5ad
output: results/mouse_tiny_notebook


## Run ACE

For speed, this notebook uses `--max-epochs 1`. Increase `--max-epochs` for a real analysis.

In [2]:
cmd = [
    sys.executable, str(Path("examples") / "mouse_aging_workflow.py"),
    "--adata", str(adata_rel),
    "--output-dir", str(out_rel),
    "--max-epochs", "1",
    "--n-salient-latent", "3",
    "--n-background-latent", "5",
    "--metric-pairs", "1000",
]
print("python " + " ".join(cmd[1:]))
completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    completed.check_returncode()
for line in completed.stdout.splitlines():
    if line.startswith("Wrote outputs"):
        print(line)

python examples/mouse_aging_workflow.py --adata data/ace_mouse_tiny.h5ad --output-dir results/mouse_tiny_notebook --max-epochs 1 --n-salient-latent 3 --n-background-latent 5 --metric-pairs 1000
Wrote outputs to results/mouse_tiny_notebook


## Outputs

In [3]:
sorted(p.name for p in out_dir.iterdir())

['ace_model',
 'age_prediction_metrics.csv',
 'background_latent.csv',
 'background_latent.h5ad',
 'salient_latent.csv',
 'salient_latent.h5ad',
 'salient_metrics_age_corr.csv']

## Age-prediction metric

A ridge regressor is fit on the training salient embedding and evaluated on held-out cells. The 1-epoch value is only a smoke-test result.

In [4]:
age_metrics = pd.read_csv(out_dir / "age_prediction_metrics.csv")
age_metrics

,n_train,n_test,r2,rmse,ridge_alpha
0,768,192,-0.015756,10.592981,10.0


## Representation metrics

These metrics are adapted from the manuscript mouse droplet evaluation. Higher age-correlation metrics indicate stronger age ordering in the salient latent space; entropy mixing metrics summarize mixing of cell traits in the salient space.

In [5]:
salient_metrics = pd.read_csv(out_dir / "salient_metrics_age_corr.csv", index_col=0)
salient_metrics

,value
gearys_c,1.019655
morans_i,-0.019738
distance_age_n_pairs,1000.000000
distance_age_spearman_r,-0.049026
distance_age_spearman_p,0.121303
distance_age_pearson_r,-0.040746
distance_age_pearson_p,0.197950
age_bin_centroid_monotonicity_spearman_r,-0.044723
geodesic_distance_age_spearman_r,-0.033722
adjacent_age_connectivity_ratio,0.319652


In [6]:
selected_metrics = salient_metrics.loc[[
    "morans_i",
    "distance_age_spearman_r",
    "geodesic_distance_age_spearman_r",
    "cell_type_AllAge_entropy_batch_mixing",
    "tissue_AllAge_entropy_batch_mixing",
    "sex_AllAge_entropy_batch_mixing",
]]
selected_metrics

,value
morans_i,-0.019738
distance_age_spearman_r,-0.049026
geodesic_distance_age_spearman_r,-0.033722
cell_type_AllAge_entropy_batch_mixing,2.129396
tissue_AllAge_entropy_batch_mixing,1.801181
sex_AllAge_entropy_batch_mixing,0.596377
